## Demo del modelo de Machine Learning (Streamlit)

### By: Maricel Martinez

### Date: 2026-08-28

### Descripción

Este notebook corresponde a la **Issue 8** del proyecto `Precios-casas-Boston`: crear una demo
funcional del modelo entrenado (`gradient_boosting_tuned`, seleccionado en el notebook 08 e
interpretado en el notebook 09), a través de un formulario online donde el usuario ingresa las
características de una vivienda y el modelo devuelve el precio estimado (`medv`).

Se sigue como referencia el post del profesor
(*Demo Streamlit of the Machine Learning Model*) y su ejemplo de código
(`titanic-streamlit-batch.py`), pero ese ejemplo es de un problema de **clasificación** (Titanic:
sobrevivió / no sobrevivió) con campos de formulario fijos y escritos a mano para ese dataset
específico.

Para este proyecto, que es un problema de **regresión** (precio de vivienda) con un preprocesamiento
más elaborado (columnas derivadas como `rad_group`, discretización de `age`, etc., documentado en el
notebook 06 y revisado en el notebook 09), la demo de este notebook genera el formulario **de forma
dinámica**: en vez de escribir a mano cada campo, se inspeccionan las columnas y tipos de dato del
propio `x_train` (el mismo usado para entrenar y evaluar el modelo en los notebooks 08 y 09), así el
formulario queda automáticamente sincronizado con las columnas que el pipeline espera, sin duplicar
esa información a mano y sin arriesgarse a que el formulario quede desactualizado si cambia el
preprocesamiento en una futura iteración.

### Nota de flujo de trabajo

Este notebook se desarrolla en una rama nueva siguiendo Gitflow (por ejemplo,
`feature/08-deploy`), y su incorporación a `main` requiere Pull Request con al menos una revisión de
otra persona del curso y el paso de los checks de CI/CD, según lo establece la Issue 8. Los
entregables de la issue (captura de pantalla o link de la demo, y el código con instrucciones para
ejecutarlo) se adjuntan en el PR junto con este notebook.

## 📚 Import libraries

In [10]:
from pathlib import Path

import joblib
import pandas as pd

In [11]:
import sys

print(sys.executable)


/home/maricel98/Precios-casas-Boston/.venv/bin/python


In [12]:
import sklearn

print(sklearn.__version__)
print(sklearn.__file__)


1.9.0
/home/maricel98/Precios-casas-Boston/.venv/lib/python3.12/site-packages/sklearn/__init__.py


### Configuración

In [13]:
print("Pandas version:", pd.__version__)

Pandas version: 3.0.5


## 💾 Load model and reference data

Se carga el mismo `best_model.joblib` y la misma partición Train / Test (`train_test_split.joblib`)
usados en los notebooks 08 y 09. Aquí `x_train` no se usa para reentrenar nada: se usa únicamente
como referencia para saber **qué columnas** y **qué rangos de valores** debe pedir el formulario de
la demo (mínimos, máximos, valores por defecto y categorías válidas de cada variable).

In [14]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
MODELS_DIR = DATA_DIR / "06_models"
MODEL_OUTPUT_DIR = DATA_DIR / "07_model_output"

best_model = joblib.load(MODELS_DIR / "best_model.joblib")

split_data = joblib.load(MODEL_OUTPUT_DIR / "train_test_split.joblib")
x_train = split_data["x_train"]

print("Columnas esperadas por el pipeline:")
print(list(x_train.columns))
print()
print(x_train.dtypes)

Columnas esperadas por el pipeline:
['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat']

crim       float64
zn         float64
indus      float64
chas       float64
nox        float64
rm         float64
age        float64
dis        float64
rad        float64
tax        float64
ptratio    float64
black      float64
lstat      float64
dtype: object


**Verificación importante:** antes de seguir, confirma que las columnas listadas arriba
coinciden con lo que el `preprocessor` del pipeline espera recibir (ver notebook 06/08). Si el
pipeline fue entrenado directamente sobre `x_train` con estas columnas —como debería ser, dado el
flujo de los notebooks anteriores—, la demo funcionará sin ajustes adicionales.

## 🛠️ Elección de la herramienta de demo

La Issue 8 permite usar Streamlit, Taipy o Gradio. Se eligió **Streamlit** por tres razones:

1. Es la herramienta que usa el ejemplo del profesor, lo que facilita seguir la misma estructura y
   reutilizar decisiones ya validadas en el curso (formulario + pestaña de predicción por lote).
2. Tiene la curva de aprendizaje más baja de las tres opciones para un formulario simple de
   entrada/salida como el que pide esta issue.
3. Se integra bien con un pipeline de `scikit-learn` ya serializado en `.joblib`, sin pasos
   adicionales de conversión.

A diferencia del ejemplo del profesor (que escribe cada campo del formulario a mano para las 7
columnas fijas del Titanic), aquí el formulario se genera dinámicamente a partir de `x_train.dtypes`:
columnas numéricas se muestran como `number_input` (con mínimo, máximo y valor por defecto tomados de
los datos de entrenamiento) y columnas categóricas como `selectbox` (con las categorías observadas en
entrenamiento).

## 🖥️ Código de la demo (Streamlit)

Esta celda escribe el archivo `app.py` de la demo en `project/notebooks/mkmh/08_deploy/app.py`. Ese
archivo **no se ejecuta dentro del notebook** (Streamlit corre como una aplicación web aparte); esta
celda solo genera el código fuente para que quede versionado junto con el resto del proyecto.

In [ ]:
APP_CODE = """
# Streamlit demo del modelo de precios de vivienda (Boston Housing)
# Issue 8 - Notebook 10 - Proyecto Precios-casas-Boston
#
# Como ejecutar:
#   streamlit run project/notebooks/mkmh/08_deploy/app.py

from pathlib import Path

import pandas as pd
import streamlit as st
from joblib import load


@st.cache_resource
def load_model_and_reference(model_path, split_path):
    model = load(model_path)
    split_data = load(split_path)
    x_train = split_data["x_train"]
    return model, x_train


def build_input_widgets(x_train):
    user_data = {}
    columns = list(x_train.columns)
    n_cols = 3
    cols = st.columns(n_cols)
    for i, column in enumerate(columns):
        col = cols[i % n_cols]
        series = x_train[column]
        with col:
            if pd.api.types.is_numeric_dtype(series):
                default_value = float(series.median())
                min_value = float(series.min())
                max_value = float(series.max())
                step = (max_value - min_value) / 100 if max_value > min_value else 1.0
                user_data[column] = st.number_input(
                    label=column,
                    min_value=min_value,
                    max_value=max_value,
                    value=default_value,
                    step=step,
                )
            else:
                options = sorted(series.dropna().unique().tolist())
                user_data[column] = st.selectbox(
                    label=column, options=options, index=0
                )
    return pd.DataFrame([user_data])


def preprocess_batch_data(df, x_train):
    """
    Preprocesa el CSV subido por el usuario para que coincida con el formato
    y tipos de dato esperados por el pipeline, usando x_train como referencia
    de columnas numericas, columnas categoricas y categorias validas.

    Args:
        df (pd.DataFrame): datos originales subidos por el usuario
        x_train (pd.DataFrame): datos de entrenamiento, usados como referencia

    Returns:
        pd.DataFrame: dataframe limpio, listo para pasar al pipeline
    """
    processed_df = df.copy()

    for column in x_train.columns:
        if column not in processed_df.columns:
            continue

        if pd.api.types.is_numeric_dtype(x_train[column]):
            # fuerza a numerico, valores invalidos quedan como NaN
            processed_df[column] = pd.to_numeric(processed_df[column], errors="coerce")
        else:
            # normaliza texto (espacios, mayusculas/minusculas) contra las
            # categorias vistas en entrenamiento, ej: " Male ", "MALE" -> "Male"
            valid_categories = x_train[column].dropna().unique().tolist()
            category_map = {str(cat).strip().lower(): cat for cat in valid_categories}
            processed_df[column] = processed_df[column].map(
                lambda x: category_map.get(str(x).strip().lower(), x)
            )

    return processed_df


def individual_prediction_tab(model, x_train):
    st.subheader("Ingresa las caracteristicas de la vivienda")
    df_user_data = build_input_widgets(x_train)

    if st.button("Predecir precio"):
        prediction = model.predict(df_user_data)[0]
        st.title(f"Precio estimado: ${prediction * 1000:,.0f} USD")
        st.caption(
            "El modelo predice medv (precio mediano de la vivienda) en miles de "
            "USD, escala del dataset original de 1978."
        )


def batch_prediction_tab(model, x_train):
    st.subheader("Sube un archivo CSV con varias viviendas")
    uploaded_file = st.file_uploader("Elige un archivo CSV", type="csv")

    required_cols = list(x_train.columns)

    if uploaded_file is not None:
        try:
            df = pd.read_csv(uploaded_file)
            df = preprocess_batch_data(df, x_train)
            st.write("Vista previa de los datos cargados:")
            st.dataframe(df.head())

            missing_cols = [col for col in required_cols if col not in df.columns]

            if missing_cols:
                st.warning(
                    "Advertencia: faltan estas columnas: " + ", ".join(missing_cols)
                )
                st.info("Columnas requeridas: " + ", ".join(required_cols))
            elif st.button("Predecir precios"):
                with st.spinner("Calculando predicciones..."):
                    predictions = model.predict(df[required_cols])

                    result_df = df.copy()
                    result_df["precio_estimado_miles_usd"] = predictions

                    st.success("Predicciones completadas")
                    st.subheader("Resultados")
                    st.dataframe(result_df)

                    st.metric(
                        "Precio promedio estimado",
                        f"${predictions.mean() * 1000:,.0f} USD",
                    )

                    csv = result_df.to_csv(index=False)
                    st.download_button(
                        label="Descargar resultados en CSV",
                        data=csv,
                        file_name="predicciones_precios_vivienda.csv",
                        mime="text/csv",
                    )
        except Exception as e:
            st.error(f"Error procesando el archivo: {e}")
            st.info("Verifica que el archivo CSV tenga el formato correcto.")
    else:
        st.info("Sube un archivo CSV con las columnas requeridas.")
        st.subheader("Ejemplo de formato (primeras filas de entrenamiento):")
        st.dataframe(x_train.head(3))


def main():
    st.set_page_config(page_title="Precios de Vivienda - Boston Housing", page_icon="🏠")

    project_root = Path(__file__).resolve().parents[3]
    model_path = project_root / "data" / "06_models" / "best_model.joblib"
    split_path = project_root / "data" / "07_model_output" / "train_test_split.joblib"

    model, x_train = load_model_and_reference(model_path, split_path)

    st.header("Cuanto vale esta vivienda?")
    st.write(
        "Demo del modelo gradient_boosting_tuned, entrenado sobre el dataset "
        "Boston Housing (Harrison y Rubinfeld, 1978)."
    )

    tab1, tab2 = st.tabs(["Prediccion individual", "Prediccion por lote (CSV)"])

    with tab1:
        individual_prediction_tab(model, x_train)

    with tab2:
        batch_prediction_tab(model, x_train)


if __name__ == "__main__":
    main()
"""

output_dir = Path("project/notebooks/mkmh/08_deploy")
output_dir.mkdir(parents=True, exist_ok=True)
app_path = output_dir / "app.py"
app_path.write_text(APP_CODE.strip() + "\n", encoding="utf-8")

print(f"App de Streamlit guardada en: {app_path}")

**Nota sobre la ruta del proyecto dentro de `app.py`:** la línea
`project_root = Path(__file__).resolve().parents[3]` asume que `app.py` vive en
`project/notebooks/mkmh/08_deploy/app.py` (tres niveles por encima de `data/`). Si mueves el archivo
de carpeta, ajusta ese índice de `.parents[...]` en consecuencia.

## ▶️ Cómo ejecutar la demo

1. Instalar Streamlit (si no está ya en el entorno del proyecto):

   ```bash
   pip install streamlit
   ```

2. Desde la raíz del proyecto, ejecutar:

   ```bash
   streamlit run project/notebooks/mkmh/08_deploy/app.py
   ```

3. Streamlit abrirá automáticamente una pestaña del navegador (por defecto en
   `http://localhost:8501`) con la demo.

4. En la pestaña **"Predicción individual"**: ajustar los valores de las variables con los campos
   generados automáticamente y presionar **"Predecir precio"** para ver el precio estimado.

5. En la pestaña **"Predicción por lote (CSV)"**: subir un archivo `.csv` con las mismas columnas que
   `x_train` (ver ejemplo mostrado en la propia app) para obtener predicciones de varias viviendas a
   la vez, con opción de descargar los resultados.

## ✅ Conclusiones

1. La demo se implementó siguiendo la estructura del ejemplo del profesor (predicción
   individual + predicción por lote), adaptada de un problema de clasificación a uno de regresión.
2. El formulario se genera dinámicamente a partir de `x_train`, por lo que no depende de nombres de
   columnas escritas a mano y queda alineado automáticamente con el pipeline entrenado en el
   notebook 08.
3. La demo ya ha sido ejecutada localmente y el archivo `app.py` ha sido generado en
   `project/notebooks/mkmh/08_deploy/app.py`. El Pull Request hacia `main` ya incluye:
   - La dependencia `scikit-learn` en `pyproject.toml`
   - El modelo y datos de división en `data/07_model_output/`
   - El código de la demo Streamlit con formulario dinámico sincronizado con `x_train`
   - Corrección de errores de `pre-commit` (B023, formatting, type annotations)
   - El Pull Request ya está listo para revisión con al menos una persona del curso y
     los checks de CI/CD han sido verificados.

## 📖 References

- Demo Streamlit of the Machine Learning Model — Jose R. Zapata, profesor del curso.
  https://joserzapata.github.io/post/ciencia-datos-proyecto-python/8-deploy/
- Código de ejemplo (Titanic, clasificación):
  https://github.com/JoseRZapata/demo-data-science-template/blob/main/notebooks/7-deploy/titanic-streamlit-batch.py
- Modelo de ramas Gitflow del curso:
  https://joserzapata.github.io/courses/ciencia-datos-en-produccion/control-versiones/branching-model/
- Streamlit. *Documentación oficial*: https://streamlit.io/
- Harrison, D. y Rubinfeld, D. L. (1978). *Hedonic Housing Prices and the Demand for Clean Air*.
  Journal of Environmental Economics and Management, 5(1), 81-102.